# DDD17 EventSpikeSeg Training

Before running the code cell:

1. **Add Input** → attach your private `stamir/ddd17-brenet-h5` dataset.
2. Select the desired **GPU accelerator**.
3. Turn **Internet ON** the first time if SpikingJelly is not already installed.
4. Run the single code cell below.

The cell reads the six HDF5 files directly from `/kaggle/input`; it does **not**
extract or copy the dataset to `/tmp`.


In [3]:
import sys
import subprocess
from pathlib import Path

TORCH_DIR = Path("/kaggle/working/torch-cu126")

if not (TORCH_DIR / "torch").exists():
    print("Installing isolated PyTorch cu126...")

    subprocess.check_call([
        sys.executable,
        "-m", "pip", "install",
        "--target", str(TORCH_DIR),
        "--no-cache-dir",
        "torch==2.10.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ])
else:
    print("✓ Isolated cu126 PyTorch already installed")

# Verify in a NEW process, without touching this notebook kernel's torch.
code = """
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("Architectures:", torch.cuda.get_arch_list())

assert torch.version.cuda == "12.6"
assert "sm_60" in torch.cuda.get_arch_list()

x = torch.zeros(1, device="cuda")
print("✓ P100 CUDA WORKS:", x)
"""

import os

env = os.environ.copy()
env["PYTHONPATH"] = str(TORCH_DIR)
env["PYTHONNOUSERSITE"] = "1"

subprocess.run(
    [sys.executable, "-c", code],
    env=env,
    check=True,
)

✓ Isolated cu126 PyTorch already installed
Torch: 2.10.0+cu126
CUDA: 12.6
GPU: Tesla P100-PCIE-16GB
Capability: (6, 0)
Architectures: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
✓ P100 CUDA WORKS: tensor([0.], device='cuda:0')


CompletedProcess(args=['/usr/bin/python3', '-c', '\nimport torch\n\nprint("Torch:", torch.__version__)\nprint("CUDA:", torch.version.cuda)\nprint("GPU:", torch.cuda.get_device_name(0))\nprint("Capability:", torch.cuda.get_device_capability(0))\nprint("Architectures:", torch.cuda.get_arch_list())\n\nassert torch.version.cuda == "12.6"\nassert "sm_60" in torch.cuda.get_arch_list()\n\nx = torch.zeros(1, device="cuda")\nprint("✓ P100 CUDA WORKS:", x)\n'], returncode=0)

In [ ]:
#!/usr/bin/env python3
"""
Kaggle single-file launcher for the canonical EventSpikeSeg DDD17 training run.

This file vendors the current project Python modules and launches the canonical
final DDD17 configuration used by the project's `train-ddd17` service.

Kaggle Notebook B:
  1. Attach the private `ddd17-brenet-h5` dataset as Notebook Input.
  2. Select a GPU accelerator.
  3. Run this notebook cell directly.

Expected attached files somewhere under /kaggle/input:
    train_dir0.h5
    train_dir3.h5
    train_dir4.h5
    train_dir6.h5
    train_dir7.h5
    test_dir1.h5

The cell auto-detects the HDF5 dataset root and streams samples directly.
"""

# ============================================================
# USE ISOLATED P100-COMPATIBLE PYTORCH
# ============================================================

import sys

TORCH_DIR = "/kaggle/working/torch-cu126"

if TORCH_DIR not in sys.path:
    sys.path.insert(0, TORCH_DIR)

import torch

print("Training PyTorch:", torch.__version__)
print("Training CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Architectures:", torch.cuda.get_arch_list())

assert torch.version.cuda == "12.6"
assert "sm_60" in torch.cuda.get_arch_list()

from __future__ import annotations

import base64
import importlib
import os
import shutil
import math
from pathlib import Path
from dataclasses import dataclass
import subprocess
import sys
import tempfile

import numpy as np

# ---------------------------------------------------------------------------
# Canonical main-training configuration from the current project compose.yaml
# ---------------------------------------------------------------------------
ARCHITECTURE = "EventSpikeSeg"
SUITE = "final"
SEED = 17
EPOCHS = 70
BATCH_SIZE = 1
TIME_STEPS = 8
HEIGHT = 212
WIDTH = 282
TRAIN_SAMPLES = 0.80
VALIDATION_FRACTION = 0.10
LEARNING_RATE = 1e-3
CHECKPOINT_EVERY = 1
USE_AMP = True

# Change this only if auto-detection does not find your Kaggle input.
DDD17_ROOT: str | None = None

# Kaggle output persists after the run.
OUTPUT_DIR = Path("/kaggle/working/eventspikeseg_checkpoints")
BUNDLE_DIR = Path("/kaggle/working/_eventspikeseg_bundle")

_BUNDLED_MODULES = {
    'data.py': 'IiIiRXYtU2VnTmV0IERERDE3IGxvYWRlci4NCg0KS25vd24gbGF5b3V0IG9uIHRoaXMgbWFjaGluZTo6DQoNCiAgICBDOi9Vc2Vycy9zdGFtaXIvRGVza3RvcC9EYXRhc2V0L0RERDE3Lw0KICAgICAgZXZlbnRzL3t0cmFpbix0ZXN0fS8qLm5weSAgICMgc2l4LWNoYW5uZWwgZXZlbnQgc3RhdGlzdGljcw0KICAgICAgbGFiZWxzL3t0cmFpbix0ZXN0fS8qLnBuZyAgICMgY2xhc3MtaW5kZXggbWFza3MgKGlnbm9yZSA9IDI1NSkNCg0KYGB0ZXN0YGAgaXMgdGhlIG9mZmljaWFsIGhlbGQtb3V0IHNldC4gUHJldmlldyBmb2xkZXJzIHN1Y2ggYXMgYGBldmVudHNfaW1hZ2VzYGANCmFuZCBgYGxhYmVscy90ZXN0X2NvbG9yZWRgYCBhcmUgaWdub3JlZC4NCiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmltcG9ydCBqc29uDQppbXBvcnQgb3MNCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHRvcmNoDQpmcm9tIFBJTCBpbXBvcnQgSW1hZ2UNCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcg0KZnJvbSB0b3JjaC5ubiBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGDQpmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQNCg0KREREMTdfQ0xBU1NfTkFNRVMgPSAoImZsYXQiLCAiY29uc3RydWN0aW9uK3NreSIsICJvYmplY3QiLCAibmF0dXJlIiwgImh1bWFuIiwgInZlaGljbGUiKQ0KREREMTdfQ0xBU1NfQ09VTlQgPSBsZW4oREREMTdfQ0xBU1NfTkFNRVMpDQpJR05PUkVfSU5ERVggPSAyNTUNClNQTElUUyA9ICgidHJhaW4iLCAidGVzdCIpDQoNCkRFRkFVTFRfREREMTdfUk9PVCA9IFBhdGgociJDOlxVc2Vyc1xzdGFtaXJcRGVza3RvcFxEYXRhc2V0XERERDE3IikNCg0KDQpkZWYgcmVzb2x2ZV9kZGQxN19yb290KHJvb3Q6IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSkgLT4gUGF0aDoNCiAgICBpZiByb290IGlzIE5vbmU6DQogICAgICAgIHJldHVybiBERUZBVUxUX0RERDE3X1JPT1QuZXhwYW5kdXNlcigpLnJlc29sdmUoKQ0KICAgIHJldHVybiBQYXRoKHJvb3QpLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkNCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgREREMTdSZWNvcmQ6DQogICAgc3BsaXQ6IHN0cg0KICAgIGV2ZW50X3BhdGg6IFBhdGgNCiAgICBsYWJlbF9wYXRoOiBQYXRoDQogICAgYXJjaGl2ZV9pbmRleDogaW50IHwgTm9uZSA9IE5vbmUNCg0KDQpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQ0KY2xhc3MgREREMTdJbnNwZWN0aW9uOg0KICAgIHJvb3Q6IHN0cg0KICAgIHJlY29yZHM6IGRpY3Rbc3RyLCBpbnRdDQogICAgZm9ybWF0czogZGljdFtzdHIsIGludF0NCiAgICB3YXJuaW5nczogdHVwbGVbc3RyLCAuLi5dDQoNCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToNCiAgICAgICAgcmV0dXJuIHsNCiAgICAgICAgICAgICJyb290Ijogc2VsZi5yb290LA0KICAgICAgICAgICAgInJlY29yZHMiOiBzZWxmLnJlY29yZHMsDQogICAgICAgICAgICAiZm9ybWF0cyI6IHNlbGYuZm9ybWF0cywNCiAgICAgICAgICAgICJ3YXJuaW5ncyI6IGxpc3Qoc2VsZi53YXJuaW5ncyksDQogICAgICAgIH0NCg0KDQpkZWYgX2xpc3RfZmlsZXMoZGlyZWN0b3J5OiBQYXRoLCBzdWZmaXg6IHN0cikgLT4gbGlzdFtQYXRoXToNCiAgICBpZiBub3QgZGlyZWN0b3J5LmlzX2RpcigpOg0KICAgICAgICByZXR1cm4gW10NCiAgICBtYXRjaGVkOiBsaXN0W1BhdGhdID0gW10NCiAgICB3aXRoIG9zLnNjYW5kaXIoZGlyZWN0b3J5KSBhcyBlbnRyaWVzOg0KICAgICAgICBmb3IgZW50cnkgaW4gZW50cmllczoNCiAgICAgICAgICAgIGlmIGVudHJ5LmlzX2ZpbGUoZm9sbG93X3N5bWxpbmtzPUZhbHNlKSBhbmQgZW50cnkubmFtZS5sb3dlcigpLmVuZHN3aXRoKHN1ZmZpeCk6DQogICAgICAgICAgICAgICAgbWF0Y2hlZC5hcHBlbmQoUGF0aChlbnRyeS5wYXRoKSkNCiAgICByZXR1cm4gc29ydGVkKG1hdGNoZWQpDQoNCg0KZGVmIF9jYW5kaWRhdGVfbGF5b3V0X3Jvb3RzKHJvb3Q6IFBhdGgpIC0+IGxpc3RbUGF0aF06DQogICAgIiIiUmV0dXJuIGEgYm91bmRlZCBzZXQgb2Yga25vd24gRXYtU2VnTmV0IHdyYXBwZXIgcm9vdHMgd2l0aG91dCBmdWxsLXRyZWUgc2NhbnMuIiIiDQogICAgY2FuZGlkYXRlcyA9IFtyb290XQ0KICAgIGlmIHJvb3QuaXNfZGlyKCk6DQogICAgICAgIHdpdGggb3Muc2NhbmRpcihyb290KSBhcyBlbnRyaWVzOg0KICAgICAgICAgICAgZmlyc3RfbGV2ZWwgPSBbUGF0aChlbnRyeS5wYXRoKSBmb3IgZW50cnkgaW4gZW50cmllcyBpZiBlbnRyeS5pc19kaXIoZm9sbG93X3N5bWxpbmtzPUZhbHNlKV0NCiAgICAgICAgY2FuZGlkYXRlcy5leHRlbmQoZmlyc3RfbGV2ZWwpDQogICAgICAgIGZvciBkaXJlY3RvcnkgaW4gZmlyc3RfbGV2ZWw6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgd2l0aCBvcy5zY2FuZGlyKGRpcmVjdG9yeSkgYXMgZW50cmllczoNCiAgICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlcy5leHRlbmQoDQogICAgICAgICAgICAgICAgICAgICAgICBQYXRoKGVudHJ5LnBhdGgpDQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgZW50cnkgaW4gZW50cmllcw0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgZW50cnkuaXNfZGlyKGZvbGxvd19zeW1saW5rcz1GYWxzZSkNCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBlbnRyeS5uYW1lLmxvd2VyKCkgaW4geyJvdXJfY29kaWZpY2F0aW9uIiwgImRkZDE3IiwgImRhdGFzZXQifQ0KICAgICAgICAgICAgICAgICAgICApDQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgIHJldHVybiBjYW5kaWRhdGVzDQoNCg0KZGVmIF9wYWlyZWRfZGlyZWN0b3J5X3JlY29yZHMobGF5b3V0X3Jvb3Q6IFBhdGgsIHNwbGl0OiBzdHIpIC0+IGxpc3RbREREMTdSZWNvcmRdOg0KICAgIGRpcmVjdG9yeV9wYWlycyA9ICgNCiAgICAgICAgKGxheW91dF9yb290IC8gImV2ZW50cyIgLyBzcGxpdCwgbGF5b3V0X3Jvb3QgLyAibGFiZWxzIiAvIHNwbGl0KSwNCiAgICAgICAgKGxheW91dF9yb290IC8gc3BsaXQgLyAiZXZlbnRzIiwgbGF5b3V0X3Jvb3QgLyBzcGxpdCAvICJsYWJlbHMiKSwNCiAgICApDQogICAgZm9yIGV2ZW50c19kaXIsIGxhYmVsc19kaXIgaW4gZGlyZWN0b3J5X3BhaXJzOg0KICAgICAgICBldmVudF9maWxlcyA9IF9saXN0X2ZpbGVzKGV2ZW50c19kaXIsICIubnB5IikNCiAgICAgICAgbGFiZWxfbGlzdCA9IF9saXN0X2ZpbGVzKGxhYmVsc19kaXIsICIucG5nIikNCiAgICAgICAgaWYgbm90IGV2ZW50X2ZpbGVzIG9yIG5vdCBsYWJlbF9saXN0Og0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgYnlfc3RlbSA9IHtwYXRoLnN0ZW06IHBhdGggZm9yIHBhdGggaW4gbGFiZWxfbGlzdH0NCiAgICAgICAgZXhhY3QgPSBbDQogICAgICAgICAgICBEREQxN1JlY29yZChzcGxpdCwgZXZlbnRfcGF0aCwgYnlfc3RlbVtldmVudF9wYXRoLnN0ZW1dKQ0KICAgICAgICAgICAgZm9yIGV2ZW50X3BhdGggaW4gZXZlbnRfZmlsZXMNCiAgICAgICAgICAgIGlmIGV2ZW50X3BhdGguc3RlbSBpbiBieV9zdGVtDQogICAgICAgIF0NCiAgICAgICAgaWYgbGVuKGV4YWN0KSA9PSBsZW4oZXZlbnRfZmlsZXMpOg0KICAgICAgICAgICAgcmV0dXJuIGV4YWN0DQogICAgICAgIGlmIGxlbihldmVudF9maWxlcykgPT0gbGVuKGxhYmVsX2xpc3QpOg0KICAgICAgICAgICAgcmV0dXJuIFsNCiAgICAgICAgICAgICAgICBEREQxN1JlY29yZChzcGxpdCwgZXZlbnRfcGF0aCwgbGFiZWxfcGF0aCkNCiAgICAgICAgICAgICAgICBmb3IgZXZlbnRfcGF0aCwgbGFiZWxfcGF0aCBpbiB6aXAoZXZlbnRfZmlsZXMsIGxhYmVsX2xpc3QsIHN0cmljdD1UcnVlKQ0KICAgICAgICAgICAgXQ0KICAgIHJldHVybiBbXQ0KDQoNCmRlZiBkaXNjb3Zlcl9kZGQxN19yZWNvcmRzKA0KICAgIHJvb3Q6IFBhdGggfCBzdHIgfCBOb25lID0gTm9uZSwNCiAgICBzcGxpdDogc3RyID0gInRyYWluIiwNCikgLT4gbGlzdFtEREQxN1JlY29yZF06DQogICAgaWYgc3BsaXQgbm90IGluIFNQTElUUzoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInNwbGl0IG11c3QgYmUgb25lIG9mIHtTUExJVFN9LCBnb3Qge3NwbGl0IXJ9IikNCiAgICByb290ID0gcmVzb2x2ZV9kZGQxN19yb290KHJvb3QpDQogICAgaWYgbm90IHJvb3QuaXNfZGlyKCk6DQogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiREREMTcgcm9vdCBkb2VzIG5vdCBleGlzdDoge3Jvb3R9IikNCg0KICAgIGZvciBsYXlvdXRfcm9vdCBpbiBfY2FuZGlkYXRlX2xheW91dF9yb290cyhyb290KToNCiAgICAgICAgcmVjb3JkcyA9IF9wYWlyZWRfZGlyZWN0b3J5X3JlY29yZHMobGF5b3V0X3Jvb3QsIHNwbGl0KQ0KICAgICAgICBpZiByZWNvcmRzOg0KICAgICAgICAgICAgcmV0dXJuIHJlY29yZHMNCg0KICAgIGFyY2hpdmUgPSByb290IC8gZiJkZGQxN197c3BsaXR9Lm5weiINCiAgICBpZiBhcmNoaXZlLmlzX2ZpbGUoKToNCiAgICAgICAgd2l0aCBucC5sb2FkKGFyY2hpdmUsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgcGF5bG9hZDoNCiAgICAgICAgICAgIGlmICJldmVudHMiIG5vdCBpbiBwYXlsb2FkIG9yICJsYWJlbHMiIG5vdCBpbiBwYXlsb2FkOg0KICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7YXJjaGl2ZX0gbXVzdCBjb250YWluIGV2ZW50cyBhbmQgbGFiZWxzIGFycmF5cyIpDQogICAgICAgICAgICBjb3VudCA9IG1pbihsZW4ocGF5bG9hZFsiZXZlbnRzIl0pLCBsZW4ocGF5bG9hZFsibGFiZWxzIl0pKQ0KICAgICAgICByZXR1cm4gW0RERDE3UmVjb3JkKHNwbGl0LCBhcmNoaXZlLCBhcmNoaXZlLCBpbmRleCkgZm9yIGluZGV4IGluIHJhbmdlKGNvdW50KV0NCiAgICByZXR1cm4gW10NCg0KDQpkZWYgaW5zcGVjdF9kZGQxNyhyb290OiBQYXRoIHwgc3RyIHwgTm9uZSA9IE5vbmUpIC0+IERERDE3SW5zcGVjdGlvbjoNCiAgICByb290ID0gcmVzb2x2ZV9kZGQxN19yb290KHJvb3QpDQogICAgY291bnRzOiBkaWN0W3N0ciwgaW50XSA9IHt9DQogICAgZm9ybWF0czogZGljdFtzdHIsIGludF0gPSB7fQ0KICAgIHdhcm5pbmdzOiBsaXN0W3N0cl0gPSBbXQ0KICAgIHRyYWluX3JlY29yZHM6IGxpc3RbREREMTdSZWNvcmRdID0gW10NCg0KICAgIGZvciBzcGxpdCBpbiBTUExJVFM6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIHJlY29yZHMgPSBkaXNjb3Zlcl9kZGQxN19yZWNvcmRzKHJvb3QsIHNwbGl0KQ0KICAgICAgICBleGNlcHQgRmlsZU5vdEZvdW5kRXJyb3I6DQogICAgICAgICAgICByZWNvcmRzID0gW10NCiAgICAgICAgaWYgc3BsaXQgPT0gInRyYWluIjoNCiAgICAgICAgICAgIHRyYWluX3JlY29yZHMgPSByZWNvcmRzDQogICAgICAgIGNvdW50c1tzcGxpdF0gPSBsZW4ocmVjb3JkcykNCiAgICAgICAgZm9yIHJlY29yZCBpbiByZWNvcmRzWzo2NF06DQogICAgICAgICAgICBzdWZmaXggPSByZWNvcmQuZXZlbnRfcGF0aC5zdWZmaXgubG93ZXIoKQ0KICAgICAgICAgICAgZm9ybWF0c1tzdWZmaXhdID0gZm9ybWF0cy5nZXQoc3VmZml4LCAwKSArIDENCg0KICAgIGlmIGNvdW50c1sidHJhaW4iXSA9PSAwOg0KICAgICAgICBpZiAocm9vdCAvICJldmVudHMiIC8gInRyYWluIikuaXNfZGlyKCkgYW5kIG5vdCAocm9vdCAvICJsYWJlbHMiIC8gInRyYWluIikuaXNfZGlyKCk6DQogICAgICAgICAgICB3YXJuaW5ncy5hcHBlbmQoImV2ZW50cy90cmFpbiBleGlzdHMgYnV0IGxhYmVscy90cmFpbiBpcyBtaXNzaW5nLiIpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICB3YXJuaW5ncy5hcHBlbmQoIk5vIHRyYWluIHJlY29yZHMgZm91bmQgdW5kZXIgZXZlbnRzL3RyYWluICsgbGFiZWxzL3RyYWluLiIpDQogICAgaWYgY291bnRzWyJ0ZXN0Il0gPT0gMDoNCiAgICAgICAgd2FybmluZ3MuYXBwZW5kKCJObyB0ZXN0IHJlY29yZHMgZm91bmQgdW5kZXIgZXZlbnRzL3Rlc3QgKyBsYWJlbHMvdGVzdC4iKQ0KICAgIGVsc2U6DQogICAgICAgIHdhcm5pbmdzLmFwcGVuZCgiSGVsZC1vdXQgZXZhbHVhdGlvbiB1c2VzIHRoZSBvZmZpY2lhbCBFdi1TZWdOZXQgdGVzdCBzcGxpdC4iKQ0KDQogICAgaWYgdHJhaW5fcmVjb3JkczoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgZXZlbnRzLCBfID0gX2xvYWRfcmVjb3JkX2FycmF5cyh0cmFpbl9yZWNvcmRzWzBdKQ0KICAgICAgICAgICAgc2hhcGUgPSB0dXBsZShldmVudHMuc2hhcGUpDQogICAgICAgICAgICBpZiBldmVudHMubmRpbSA9PSAzIGFuZCAoc2hhcGVbMF0gPT0gNiBvciBzaGFwZVstMV0gPT0gNik6DQogICAgICAgICAgICAgICAgd2FybmluZ3MuYXBwZW5kKA0KICAgICAgICAgICAgICAgICAgICAiRXYtU2VnTmV0IHNpeC1jaGFubmVsIHJlcHJlc2VudGF0aW9uIGRldGVjdGVkOiB0aGUgZXhwZWN0ZWQgcHJpbWFyeSBpbnB1dCAiDQogICAgICAgICAgICAgICAgICAgICJjb250YWlucyB0d28gaGlzdG9ncmFtcywgdHdvIHRpbWVzdGFtcCBtZWFucywgYW5kIHR3byB0aW1lc3RhbXAgc3RhbmRhcmQgZGV2aWF0aW9ucy4iDQogICAgICAgICAgICAgICAgKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICB3YXJuaW5ncy5hcHBlbmQoZiJVbmV4cGVjdGVkIGV2ZW50IHNoYXBlIGZvciBmaXJzdCB0cmFpbiBzYW1wbGU6IHtzaGFwZX0iKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoNCiAgICAgICAgICAgIHdhcm5pbmdzLmFwcGVuZChmIkNvdWxkIG5vdCBsb2FkIGZpcnN0IHRyYWluIHNhbXBsZToge2V4Y30iKQ0KDQogICAgcmV0dXJuIERERDE3SW5zcGVjdGlvbihzdHIocm9vdCksIGNvdW50cywgZm9ybWF0cywgdHVwbGUod2FybmluZ3MpKQ0KDQoNCmRlZiBfbG9hZF9yZWNvcmRfYXJyYXlzKHJlY29yZDogREREMTdSZWNvcmQpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOg0KICAgIGlmIHJlY29yZC5hcmNoaXZlX2luZGV4IGlzIG5vdCBOb25lOg0KICAgICAgICB3aXRoIG5wLmxvYWQocmVjb3JkLmV2ZW50X3BhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkgYXMgcGF5bG9hZDoNCiAgICAgICAgICAgIHJldHVybiAoDQogICAgICAgICAgICAgICAgbnAuYXNhcnJheShwYXlsb2FkWyJldmVudHMiXVtyZWNvcmQuYXJjaGl2ZV9pbmRleF0pLA0KICAgICAgICAgICAgICAgIG5wLmFzYXJyYXkocGF5bG9hZFsibGFiZWxzIl1bcmVjb3JkLmFyY2hpdmVfaW5kZXhdKSwNCiAgICAgICAgICAgICkNCiAgICBldmVudHMgPSBucC5hc2FycmF5KG5wLmxvYWQocmVjb3JkLmV2ZW50X3BhdGgsIGFsbG93X3BpY2tsZT1GYWxzZSkpDQogICAgbGFiZWxzID0gbnAuYXNhcnJheShJbWFnZS5vcGVuKHJlY29yZC5sYWJlbF9wYXRoKSkNCiAgICByZXR1cm4gZXZlbnRzLCBsYWJlbHMNCg0KDQpkZWYgX2Nhbm9uaWNhbF9ldmVudHMoYXJyYXk6IG5wLm5kYXJyYXkpIC0+IFRlbnNvcjoNCiAgICAiIiJSZXR1cm4gb25lIEV2LVNlZ05ldCBzYW1wbGUgYXMgYGBbMSwgNiwgSCwgV11gYC4iIiINCg0KICAgIHRlbnNvciA9IHRvcmNoLmFzX3RlbnNvcihhcnJheSkuZmxvYXQoKS5zcXVlZXplKCkNCiAgICBpZiB0ZW5zb3IubmRpbSAhPSAzOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhwZWN0ZWQgYSAzLUQgRXYtU2VnTmV0IGV2ZW50IGltYWdlLCBnb3Qge3R1cGxlKHRlbnNvci5zaGFwZSl9IikNCiAgICBpZiB0ZW5zb3Iuc2hhcGVbMF0gPT0gNjoNCiAgICAgICAgY2hhbm5lbHNfZmlyc3QgPSB0ZW5zb3INCiAgICBlbGlmIHRlbnNvci5zaGFwZVstMV0gPT0gNjoNCiAgICAgICAgY2hhbm5lbHNfZmlyc3QgPSB0ZW5zb3IucGVybXV0ZSgyLCAwLCAxKQ0KICAgIGVsc2U6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoDQogICAgICAgICAgICAiRXYtU2VnTmV0IHNhbXBsZXMgbXVzdCBjb250YWluIHNpeCBjaGFubmVscyAiDQogICAgICAgICAgICBmIihoaXN0b2dyYW0vbWVhbi9zdGQgZm9yIHR3byBwb2xhcml0aWVzKTsgZ290IHt0dXBsZSh0ZW5zb3Iuc2hhcGUpfSINCiAgICAgICAgKQ0KICAgIHJldHVybiB0b3JjaC5uYW5fdG9fbnVtKGNoYW5uZWxzX2ZpcnN0LCBuYW49MC4wLCBwb3NpbmY9MC4wLCBuZWdpbmY9MC4wKS51bnNxdWVlemUoMCkNCg0KDQpkZWYgX2Nhbm9uaWNhbF9tYXNrKGFycmF5OiBucC5uZGFycmF5KSAtPiBUZW5zb3I6DQogICAgbWFzayA9IHRvcmNoLmFzX3RlbnNvcihucC5hcnJheShhcnJheSwgY29weT1UcnVlKSkuc3F1ZWV6ZSgpLmxvbmcoKQ0KICAgIGlmIG1hc2submRpbSAhPSAyOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5zdXBwb3J0ZWQgc2VtYW50aWMgbWFzayBzaGFwZToge3R1cGxlKG1hc2suc2hhcGUpfSIpDQogICAgbWFza1sobWFzayA8IDApIHwgKG1hc2sgPj0gREREMTdfQ0xBU1NfQ09VTlQpXSA9IElHTk9SRV9JTkRFWA0KICAgIHJldHVybiBtYXNrDQoNCg0KY2xhc3MgREREMTdEYXRhc2V0KERhdGFzZXRbdHVwbGVbVGVuc29yLCBUZW5zb3JdXSk6DQogICAgZGVmIF9faW5pdF9fKA0KICAgICAgICBzZWxmLA0KICAgICAgICByb290OiBQYXRoIHwgc3RyIHwgTm9uZSA9IE5vbmUsDQogICAgICAgICosDQogICAgICAgIHNwbGl0OiBzdHIsDQogICAgICAgIG91dHB1dF9zaXplOiB0dXBsZVtpbnQsIGludF0gfCBOb25lID0gTm9uZSwNCiAgICAgICAgcmVjb3JkczogbGlzdFtEREQxN1JlY29yZF0gfCBOb25lID0gTm9uZSwNCiAgICApIC0+IE5vbmU6DQogICAgICAgIHNlbGYucm9vdCA9IHJlc29sdmVfZGRkMTdfcm9vdChyb290KQ0KICAgICAgICBzZWxmLnJlY29yZHMgPSByZWNvcmRzIGlmIHJlY29yZHMgaXMgbm90IE5vbmUgZWxzZSBkaXNjb3Zlcl9kZGQxN19yZWNvcmRzKHNlbGYucm9vdCwgc3BsaXQpDQogICAgICAgIGlmIG5vdCBzZWxmLnJlY29yZHM6DQogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIm5vIERERDE3IHtzcGxpdH0gcmVjb3JkcyB1bmRlciB7c2VsZi5yb290fSIpDQogICAgICAgIHNlbGYub3V0cHV0X3NpemUgPSBvdXRwdXRfc2l6ZQ0KDQogICAgZGVmIF9fbGVuX18oc2VsZikgLT4gaW50Og0KICAgICAgICByZXR1cm4gbGVuKHNlbGYucmVjb3JkcykNCg0KICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogaW50KSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvcl06DQogICAgICAgIGV2ZW50c19hcnJheSwgbWFza19hcnJheSA9IF9sb2FkX3JlY29yZF9hcnJheXMoc2VsZi5yZWNvcmRzW2luZGV4XSkNCiAgICAgICAgZXZlbnRzID0gX2Nhbm9uaWNhbF9ldmVudHMoZXZlbnRzX2FycmF5KQ0KICAgICAgICBtYXNrID0gX2Nhbm9uaWNhbF9tYXNrKG1hc2tfYXJyYXkpDQogICAgICAgIGlmIHNlbGYub3V0cHV0X3NpemUgaXMgbm90IE5vbmU6DQogICAgICAgICAgICB0LCBjLCBfLCBfID0gZXZlbnRzLnNoYXBlDQogICAgICAgICAgICBldmVudHMgPSBGLmludGVycG9sYXRlKA0KICAgICAgICAgICAgICAgIGV2ZW50cy52aWV3KHQgKiBjLCAxLCAqZXZlbnRzLnNoYXBlWy0yOl0pLA0KICAgICAgICAgICAgICAgIHNpemU9c2VsZi5vdXRwdXRfc2l6ZSwNCiAgICAgICAgICAgICAgICBtb2RlPSJhcmVhIiwNCiAgICAgICAgICAgICkudmlldyh0LCBjLCAqc2VsZi5vdXRwdXRfc2l6ZSkNCiAgICAgICAgICAgIG1hc2sgPSBGLmludGVycG9sYXRlKA0KICAgICAgICAgICAgICAgIG1hc2tbTm9uZSwgTm9uZV0uZmxvYXQoKSwgc2l6ZT1zZWxmLm91dHB1dF9zaXplLCBtb2RlPSJuZWFyZXN0Ig0KICAgICAgICAgICAgKVswLCAwXS5sb25nKCkNCiAgICAgICAgcmV0dXJuIGV2ZW50cywgbWFzaw0KDQoNCmRlZiBkZXRlcm1pbmlzdGljX3JlY29yZF9zdWJzZXQoDQogICAgcmVjb3JkczogbGlzdFtEREQxN1JlY29yZF0sICosIG1heGltdW06IGludCwgc2VlZDogaW50DQopIC0+IGxpc3RbREREMTdSZWNvcmRdOg0KICAgIGlmIG1heGltdW0gPD0gMCBvciBtYXhpbXVtID49IGxlbihyZWNvcmRzKToNCiAgICAgICAgcmV0dXJuIGxpc3QocmVjb3JkcykNCiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVkKQ0KICAgIGluZGljZXMgPSB0b3JjaC5yYW5kcGVybShsZW4ocmVjb3JkcyksIGdlbmVyYXRvcj1nZW5lcmF0b3IpWzptYXhpbXVtXS50b2xpc3QoKQ0KICAgIHJldHVybiBbcmVjb3Jkc1tpbmRleF0gZm9yIGluZGV4IGluIGluZGljZXNdDQoNCg0KZGVmIHdyaXRlX2luc3BlY3Rpb24ocm9vdDogUGF0aCB8IHN0ciB8IE5vbmUsIGRlc3RpbmF0aW9uOiBQYXRoKSAtPiBOb25lOg0KICAgIGRlc3RpbmF0aW9uLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgZGVzdGluYXRpb24ud3JpdGVfdGV4dChqc29uLmR1bXBzKGluc3BlY3RfZGRkMTcocm9vdCkudG9fZGljdCgpLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpDQo=',
    'metrics.py': 'IiIiU2VtYW50aWMtc2VnbWVudGF0aW9uIG1ldHJpY3MgZm9yIERERDE3LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgU2VnbWVudGF0aW9uTWV0cmljczoKICAgIG1lYW5faW91OiBmbG9hdAogICAgcGl4ZWxfYWNjdXJhY3k6IGZsb2F0CiAgICBwZXJfY2xhc3NfaW91OiB0dXBsZVtmbG9hdCwgLi4uXQogICAgdmFsaWRfY2xhc3NlczogaW50CiAgICBncm91bmRfdHJ1dGhfcGl4ZWxzOiB0dXBsZVtpbnQsIC4uLl0KICAgIHByZWRpY3RlZF9waXhlbHM6IHR1cGxlW2ludCwgLi4uXQoKICAgIGRlZiB0b19kaWN0KHNlbGYpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJtZWFuX2lvdSI6IHNlbGYubWVhbl9pb3UsCiAgICAgICAgICAgICJwaXhlbF9hY2N1cmFjeSI6IHNlbGYucGl4ZWxfYWNjdXJhY3ksCiAgICAgICAgICAgICJwZXJfY2xhc3NfaW91IjogbGlzdChzZWxmLnBlcl9jbGFzc19pb3UpLAogICAgICAgICAgICAidmFsaWRfY2xhc3NlcyI6IHNlbGYudmFsaWRfY2xhc3NlcywKICAgICAgICAgICAgImdyb3VuZF90cnV0aF9waXhlbHMiOiBsaXN0KHNlbGYuZ3JvdW5kX3RydXRoX3BpeGVscyksCiAgICAgICAgICAgICJwcmVkaWN0ZWRfcGl4ZWxzIjogbGlzdChzZWxmLnByZWRpY3RlZF9waXhlbHMpLAogICAgICAgIH0KCgpjbGFzcyBDb25mdXNpb25NYXRyaXg6CiAgICBkZWYgX19pbml0X18oc2VsZiwgY2xhc3NfY291bnQ6IGludCwgaWdub3JlX2luZGV4OiBpbnQgPSAyNTUpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5jbGFzc19jb3VudCA9IGNsYXNzX2NvdW50CiAgICAgICAgc2VsZi5pZ25vcmVfaW5kZXggPSBpZ25vcmVfaW5kZXgKICAgICAgICBzZWxmLm1hdHJpeCA9IHRvcmNoLnplcm9zKChjbGFzc19jb3VudCwgY2xhc3NfY291bnQpLCBkdHlwZT10b3JjaC5pbnQ2NCkKCiAgICBkZWYgdXBkYXRlKHNlbGYsIHByZWRpY3Rpb246IFRlbnNvciwgdGFyZ2V0OiBUZW5zb3IpIC0+IE5vbmU6CiAgICAgICAgcHJlZGljdGlvbiA9IHByZWRpY3Rpb24uZGV0YWNoKCkuY3B1KCkubG9uZygpLnJlc2hhcGUoLTEpCiAgICAgICAgdGFyZ2V0ID0gdGFyZ2V0LmRldGFjaCgpLmNwdSgpLmxvbmcoKS5yZXNoYXBlKC0xKQogICAgICAgIHZhbGlkID0gdGFyZ2V0ICE9IHNlbGYuaWdub3JlX2luZGV4CiAgICAgICAgdmFsaWQgJj0gKHRhcmdldCA+PSAwKSAmICh0YXJnZXQgPCBzZWxmLmNsYXNzX2NvdW50KQogICAgICAgIHZhbGlkICY9IChwcmVkaWN0aW9uID49IDApICYgKHByZWRpY3Rpb24gPCBzZWxmLmNsYXNzX2NvdW50KQogICAgICAgIGVuY29kZWQgPSB0YXJnZXRbdmFsaWRdICogc2VsZi5jbGFzc19jb3VudCArIHByZWRpY3Rpb25bdmFsaWRdCiAgICAgICAgc2VsZi5tYXRyaXggKz0gdG9yY2guYmluY291bnQoCiAgICAgICAgICAgIGVuY29kZWQsIG1pbmxlbmd0aD1zZWxmLmNsYXNzX2NvdW50ICogc2VsZi5jbGFzc19jb3VudAogICAgICAgICkucmVzaGFwZShzZWxmLmNsYXNzX2NvdW50LCBzZWxmLmNsYXNzX2NvdW50KQoKICAgIGRlZiBjb21wdXRlKHNlbGYpIC0+IFNlZ21lbnRhdGlvbk1ldHJpY3M6CiAgICAgICAgbWF0cml4ID0gc2VsZi5tYXRyaXguZmxvYXQoKQogICAgICAgIHRydWVfcG9zaXRpdmUgPSBtYXRyaXguZGlhZygpCiAgICAgICAgdW5pb24gPSBtYXRyaXguc3VtKGRpbT0wKSArIG1hdHJpeC5zdW0oZGltPTEpIC0gdHJ1ZV9wb3NpdGl2ZQogICAgICAgIHZhbGlkID0gdW5pb24gPiAwCiAgICAgICAgaW91ID0gdG9yY2guZnVsbCgoc2VsZi5jbGFzc19jb3VudCwpLCBmbG9hdCgibmFuIikpCiAgICAgICAgaW91W3ZhbGlkXSA9IHRydWVfcG9zaXRpdmVbdmFsaWRdIC8gdW5pb25bdmFsaWRdCiAgICAgICAgdG90YWwgPSBtYXRyaXguc3VtKCkKICAgICAgICBhY2N1cmFjeSA9IGZsb2F0KHRydWVfcG9zaXRpdmUuc3VtKCkgLyB0b3RhbCkgaWYgdG90YWwgPiAwIGVsc2UgMC4wCiAgICAgICAgbWVhbl9pb3UgPSBmbG9hdChpb3VbdmFsaWRdLm1lYW4oKSkgaWYgdmFsaWQuYW55KCkgZWxzZSAwLjAKICAgICAgICByZXR1cm4gU2VnbWVudGF0aW9uTWV0cmljcygKICAgICAgICAgICAgbWVhbl9pb3U9bWVhbl9pb3UsCiAgICAgICAgICAgIHBpeGVsX2FjY3VyYWN5PWFjY3VyYWN5LAogICAgICAgICAgICBwZXJfY2xhc3NfaW91PXR1cGxlKGZsb2F0KHZhbHVlKSBmb3IgdmFsdWUgaW4gaW91KSwKICAgICAgICAgICAgdmFsaWRfY2xhc3Nlcz1pbnQodmFsaWQuc3VtKCkpLAogICAgICAgICAgICBncm91bmRfdHJ1dGhfcGl4ZWxzPXR1cGxlKGludCh2YWx1ZSkgZm9yIHZhbHVlIGluIHNlbGYubWF0cml4LnN1bShkaW09MSkpLAogICAgICAgICAgICBwcmVkaWN0ZWRfcGl4ZWxzPXR1cGxlKGludCh2YWx1ZSkgZm9yIHZhbHVlIGluIHNlbGYubWF0cml4LnN1bShkaW09MCkpLAogICAgICAgICkK',
    'model_components.py': '...',
    'model.py': '...',
    'top_ten_models.py': 'IiIiRnJvemVuIGRlZmluaXRpb25zIGZyb20gdGhlIGNvbXBsZXRlZCB0ZW4tbW9kZWwgREREMTcgYmVuY2htYXJrLiIiIg0KDQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zDQoNCmZyb20gLm1vZGVsX2NvbXBvbmVudHMgaW1wb3J0IEV2ZW50U3Bpa2VTZWdCYXNlLCBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmUNCg0KVE9QX1RFTl9QUkVTRVRTOiBkaWN0W3N0ciwgU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlXSA9IHsNCiAgICAidjAxX3N0ZF91bmV0X2xpZ2h0IjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MDFfc3RkX3VuZXRfbGlnaHQiLA0KICAgICAgICBkZXB0aHdpc2U9RmFsc2UsDQogICAgICAgIGNvbnRleHQ9Im5vbmUiLA0KICAgICAgICBub3JtPSJub25lIiwNCiAgICAgICAgYXV4aWxpYXJ5X2hlYWQ9RmFsc2UsDQogICAgICAgIHJlc2lkdWFsPUZhbHNlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9RmFsc2UsDQogICAgKSwNCiAgICAidjAyX2R3X2xpZ2h0IjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MDJfZHdfbGlnaHQiLA0KICAgICAgICBjb250ZXh0PSJub25lIiwNCiAgICAgICAgbm9ybT0ibm9uZSIsDQogICAgICAgIHJlc2lkdWFsPUZhbHNlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9RmFsc2UsDQogICAgKSwNCiAgICAidjAzX2R3X2RpbGF0ZWQiOiBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmUoDQogICAgICAgIG5hbWU9InYwM19kd19kaWxhdGVkIiwNCiAgICAgICAgY29udGV4dD0iZGlsYXRlZCIsDQogICAgICAgIGNvbnRleHRfZGlsYXRpb25zPSgyLCksDQogICAgICAgIG5vcm09Im5vbmUiLA0KICAgICAgICByZXNpZHVhbD1UcnVlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9RmFsc2UsDQogICAgKSwNCiAgICAidjA0X2R3X211bHRpc2NhbGUiOiBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmUoDQogICAgICAgIG5hbWU9InYwNF9kd19tdWx0aXNjYWxlIiwNCiAgICAgICAgY29udGV4dD0ibXVsdGlzY2FsZSIsDQogICAgICAgIG5vcm09Im5vbmUiLA0KICAgICAgICByZXNpZHVhbD1UcnVlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9RmFsc2UsDQogICAgKSwNCiAgICAidjA1X2R3X211bHRpc2NhbGVfYWZmaW5lIjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MDVfZHdfbXVsdGlzY2FsZV9hZmZpbmUiLA0KICAgICAgICBjb250ZXh0PSJtdWx0aXNjYWxlIiwNCiAgICAgICAgbm9ybT0iYWZmaW5lIiwNCiAgICAgICAgcmVzaWR1YWw9VHJ1ZSwNCiAgICAgICAgYWRhcHRpdmVfdGhyZXNob2xkPUZhbHNlLA0KICAgICksDQogICAgInYwNl9kd19tdWx0aXNjYWxlX2JuIjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MDZfZHdfbXVsdGlzY2FsZV9ibiIsDQogICAgICAgIGNvbnRleHQ9Im11bHRpc2NhbGUiLA0KICAgICAgICBub3JtPSJibiIsDQogICAgICAgIHJlc2lkdWFsPVRydWUsDQogICAgICAgIGFkYXB0aXZlX3RocmVzaG9sZD1GYWxzZSwNCiAgICApLA0KICAgICJ2MDdfZHdfbXVsdGlzY2FsZV9nYXAiOiBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmUoDQogICAgICAgIG5hbWU9InYwN19kd19tdWx0aXNjYWxlX2dhcCIsDQogICAgICAgIGNvbnRleHQ9Im11bHRpc2NhbGUiLA0KICAgICAgICBub3JtPSJhZmZpbmUiLA0KICAgICAgICBnYXBfZ2F0ZT1UcnVlLA0KICAgICAgICByZXNpZHVhbD1UcnVlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9RmFsc2UsDQogICAgKSwNCiAgICAidjA4X2R3X211bHRpc2NhbGVfYXV4IjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MDhfZHdfbXVsdGlzY2FsZV9hdXgiLA0KICAgICAgICBjb250ZXh0PSJtdWx0aXNjYWxlIiwNCiAgICAgICAgbm9ybT0iYWZmaW5lIiwNCiAgICAgICAgYXV4aWxpYXJ5X2hlYWQ9VHJ1ZSwNCiAgICAgICAgcmVzaWR1YWw9VHJ1ZSwNCiAgICAgICAgYWRhcHRpdmVfdGhyZXNob2xkPUZhbHNlLA0KICAgICksDQogICAgInYwOV9kd19zaW5nbGVfc2tpcCI6IFNlZ21lbnRhdGlvbkFyY2hpdGVjdHVyZSgNCiAgICAgICAgbmFtZT0idjA5X2R3X3NpbmdsZV9za2lwIiwNCiAgICAgICAgY29udGV4dD0ibXVsdGlzY2FsZSIsDQogICAgICAgIG5vcm09ImFmZmluZSIsDQogICAgICAgIHNraXBfa2luZD0ib25lIiwNCiAgICAgICAgcmVzaWR1YWw9VHJ1ZSwNCiAgICAgICAgYWRhcHRpdmVfdGhyZXNob2xkPVRydWUsDQogICAgKSwNCiAgICAidjEwX2ZwZ2FfaHlicmlkIjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJ2MTBfZnBnYV9oeWJyaWQiLA0KICAgICAgICBjb250ZXh0PSJtdWx0aXNjYWxlIiwNCiAgICAgICAgbm9ybT0iYWZmaW5lIiwNCiAgICAgICAgYXV4aWxpYXJ5X2hlYWQ9VHJ1ZSwNCiAgICAgICAgc2tpcF9raW5kPSJ0d28iLA0KICAgICAgICByZXNpZHVhbD1UcnVlLA0KICAgICAgICBhZGFwdGl2ZV90aHJlc2hvbGQ9VHJ1ZSwNCiAgICApLA0KfQ0KDQpUT1BfVEVOX0RFRkFVTFQgPSAidjEwX2ZwZ2FfaHlicmlkIg0KDQoNCmRlZiBidWlsZF90b3BfdGVuX21vZGVsKA0KICAgIG5hbWU6IHN0ciA9IFRPUF9URU5fREVGQVVMVCwNCiAgICAqLA0KICAgIGNsYXNzX2NvdW50OiBpbnQgPSA2LA0KICAgIGlucHV0X2NoYW5uZWxzOiBpbnQgPSA2LA0KKSAtPiBFdmVudFNwaWtlU2VnQmFzZToNCiAgICBpZiBuYW1lIG5vdCBpbiBUT1BfVEVOX1BSRVNFVFM6DQogICAgICAgIHJhaXNlIEtleUVycm9yKGYidW5rbm93biB0b3AtdGVuIGFyY2hpdGVjdHVyZSB7bmFtZSFyfSIpDQogICAgcmV0dXJuIEV2ZW50U3Bpa2VTZWdCYXNlKA0KICAgICAgICBjbGFzc19jb3VudD1jbGFzc19jb3VudCwNCiAgICAgICAgaW5wdXRfY2hhbm5lbHM9aW5wdXRfY2hhbm5lbHMsDQogICAgICAgIGFyY2hpdGVjdHVyZT1UT1BfVEVOX1BSRVNFVFNbbmFtZV0sDQogICAgKQ0K',
    'top_five_models.py': 'IiIiU3RydWN0dXJhbCB0b3AtZml2ZSBEREQxNyBTTk4gc2VnbWVudGF0aW9uIGNvbnRlbmRlcnMuDQoNClRoZSBjb21wbGV0ZWQgbWVjaGFuaXNtIGNvbXBhcmlzb24gc2VsZWN0ZWQgb25lIGNvbW1vbiBuZXVyb21vcnBoaWMgb3BlcmF0aW5nIHBvaW50IGZvcg0KYWxsIGZpdmUgY2FuZGlkYXRlczogUExJRiArIGxlYXJuYWJsZSB0aHJlc2hvbGRzLCBzdGVtLW9ubHkgcGVlci1tZW1icmFuZSBpbmhpYml0aW9uLA0KZW5jb2RlciB0cmlhbmd1bGFyIHRocmVzaG9sZCBhZGFwdGF0aW9uLCBhZmZpbmUgY2hhbm5lbCBzY2FsaW5nLCByZXNpZHVhbCBwYXRocywNCmF1eGlsaWFyeSBzdXBlcnZpc2lvbiwgYW5kIG91dHB1dCBjbGFzcyB3aW5uZXItdGFrZS1hbGwuIFRoaXMgcm91bmQgdmFyaWVzIG9ubHkgdGhlDQpzdHJ1Y3R1cmFsIGF4ZXM6IHdpZHRoLCBlYXJseSByZWNlcHRpdmUgZmllbGQsIGVuY29kZXIgZGVwdGgsIGFuZCBjb250ZXh0IHN0cmF0ZWd5Lg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMNCg0KZnJvbSAubW9kZWxfY29tcG9uZW50cyBpbXBvcnQgRXZlbnRTcGlrZVNlZ0Jhc2UsIFNlZ21lbnRhdGlvbkFyY2hpdGVjdHVyZQ0KDQoNCl9DT01NT05fRFlOQU1JQ1M6IGRpY3Rbc3RyLCBvYmplY3RdID0gew0KICAgICJub3JtIjogImFmZmluZSIsDQogICAgImF1eGlsaWFyeV9oZWFkIjogVHJ1ZSwNCiAgICAic2tpcF9raW5kIjogInR3byIsDQogICAgInJlc2lkdWFsIjogVHJ1ZSwNCiAgICAiYWRhcHRpdmVfdGhyZXNob2xkIjogRmFsc2UsDQogICAgIm5ldXJvbl9raW5kIjogInBsaWYiLA0KICAgICJsZWFybmFibGVfdGhyZXNob2xkIjogVHJ1ZSwNCiAgICAiZl9pbnN0X2VuYWJsZWQiOiBUcnVlLA0KICAgICJmX2luc3Rfc2NvcGUiOiAic3RlbSIsDQogICAgImluaGliaXRpb25fa2luZCI6ICJwZWVyX21lbWJyYW5lIiwNCiAgICAiaW5oaWJpdGlvbl9zdHJlbmd0aF9pbml0IjogMC4yMCwNCiAgICAiZl9sb25nX2VuYWJsZWQiOiBUcnVlLA0KICAgICJmX2xvbmdfc2NvcGUiOiAiZW5jb2RlciIsDQogICAgImhvbWVvc3Rhc2lzX2tpbmQiOiAidHJpYW5ndWxhcl90aHJlc2hvbGQiLA0KICAgICJ0aHJlc2hvbGRfYWRhcHRhdGlvbl9zdHJlbmd0aF9pbml0IjogMC4yMCwNCiAgICAiYWRhcHRhdGlvbl9kdXJhdGlvbl9pbml0IjogNS4wLA0KICAgICJvdXRwdXRfZl9pbnN0X2VuYWJsZWQiOiBUcnVlLA0KfQ0KDQoNClRPUF9GSVZFX1BSRVNFVFM6IGRpY3Rbc3RyLCBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmVdID0gew0KICAgICMgQ29udHJvbGxlZCAyNC80OC85NiBiYXNlbGluZSB1c2luZyB0aGUgc3Ryb25nZXN0IHJvdW5kLW9uZSBjb250ZXh0IGZhbWlseS4NCiAgICAiZjAxX2Jhc2UyNF9tdWx0aXNjYWxlIjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJmMDFfYmFzZTI0X211bHRpc2NhbGUiLA0KICAgICAgICBjaGFubmVsX3NjaGVkdWxlPSgyNCwgNDgsIDk2KSwNCiAgICAgICAgY29udGV4dD0ibXVsdGlzY2FsZSIsDQogICAgICAgIGNvbnRleHRfZGlsYXRpb25zPSgxLCAyLCA0KSwNCiAgICAgICAgKipfQ09NTU9OX0RZTkFNSUNTLA0KICAgICksDQogICAgIyBDYXBhY2l0eSB0ZXN0OiBzYW1lIGRlcHRoIGFuZCBrZXJuZWxzLCB3aWRlciBmZWF0dXJlIG1hcHMuDQogICAgImYwMl93aWRlMzJfbXVsdGlzY2FsZSI6IFNlZ21lbnRhdGlvbkFyY2hpdGVjdHVyZSgNCiAgICAgICAgbmFtZT0iZjAyX3dpZGUzMl9tdWx0aXNjYWxlIiwNCiAgICAgICAgY2hhbm5lbF9zY2hlZHVsZT0oMzIsIDY0LCAxMjgpLA0KICAgICAgICBjb250ZXh0PSJtdWx0aXNjYWxlIiwNCiAgICAgICAgY29udGV4dF9kaWxhdGlvbnM9KDEsIDIsIDQpLA0KICAgICAgICAqKl9DT01NT05fRFlOQU1JQ1MsDQogICAgKSwNCiAgICAjIFJlY2VwdGl2ZS1maWVsZCB0ZXN0OiA1eDUgb25seSB3aGVyZSByYXcvbG9jYWwgZXZlbnQgc3RydWN0dXJlIGZpcnN0IGVudGVycy4NCiAgICAiZjAzX2Vhcmx5NXg1X211bHRpc2NhbGUiOiBTZWdtZW50YXRpb25BcmNoaXRlY3R1cmUoDQogICAgICAgIG5hbWU9ImYwM19lYXJseTV4NV9tdWx0aXNjYWxlIiwNCiAgICAgICAgY2hhbm5lbF9zY2hlZHVsZT0oMjQsIDQ4LCA5NiksDQogICAgICAgIHN0ZW1fa2VybmVsX3NpemU9NSwNCiAgICAgICAgZmlyc3RfYmxvY2tfa2VybmVsX3NpemU9NSwNCiAgICAgICAgY29udGV4dD0ibXVsdGlzY2FsZSIsDQogICAgICAgIGNvbnRleHRfZGlsYXRpb25zPSgxLCAyLCA0KSwNCiAgICAgICAgKipfQ09NTU9OX0RZTkFNSUNTLA0KICAgICksDQogICAgIyBEZXB0aC9jb250ZXh0IHRlc3Q6IG9uZSBleHRyYSBlbmNvZGVyIHNjYWxlLCB0aGVuIGEgc2luZ2xlIGNoZWFwIGRpbGF0ZWQgYnJhbmNoLg0KICAgICJmMDRfZGVlcDRfZGlsYXRlZCI6IFNlZ21lbnRhdGlvbkFyY2hpdGVjdHVyZSgNCiAgICAgICAgbmFtZT0iZjA0X2RlZXA0X2RpbGF0ZWQiLA0KICAgICAgICBjaGFubmVsX3NjaGVkdWxlPSgyNCwgNDgsIDk2LCAxMjgpLA0KICAgICAgICBjb250ZXh0PSJkaWxhdGVkIiwNCiAgICAgICAgY29udGV4dF9kaWxhdGlvbnM9KDIsKSwNCiAgICAgICAgKipfQ09NTU9OX0RZTkFNSUNTLA0KICAgICksDQogICAgIyBGUEdBLWVmZmljaWVuY3kgdGVzdDogbmFycm93IGNoYW5uZWxzIHBsdXMgdGhlIGNoZWFwZXIgc2luZ2xlIGRpbGF0ZWQgY29udGV4dC4NCiAgICAiZjA1X2NvbXBhY3QxNl9kaWxhdGVkIjogU2VnbWVudGF0aW9uQXJjaGl0ZWN0dXJlKA0KICAgICAgICBuYW1lPSJmMDVfY29tcGFjdDE2X2RpbGF0ZWQiLA0KICAgICAgICBjaGFubmVsX3NjaGVkdWxlPSgxNiwgMzIsIDY0KSwNCiAgICAgICAgY29udGV4dD0iZGlsYXRlZCIsDQogICAgICAgIGNvbnRleHRfZGlsYXRpb25zPSgyLCksDQogICAgICAgICoqX0NPTU1PTl9EWU5BTUlDUywNCiAgICApLA0KfQ0KDQpUT1BfRklWRV9ERUZBVUxUID0gImYwMV9iYXNlMjRfbXVsdGlzY2FsZSINCg0KDQpkZWYgYnVpbGRfdG9wX2ZpdmVfbW9kZWwoDQogICAgbmFtZTogc3RyID0gVE9QX0ZJVkVfREVGQVVMVCwNCiAgICAqLA0KICAgIGNsYXNzX2NvdW50OiBpbnQgPSA2LA0KICAgIGlucHV0X2NoYW5uZWxzOiBpbnQgPSA2LA0KKSAtPiBFdmVudFNwaWtlU2VnQmFzZToNCiAgICBpZiBuYW1lIG5vdCBpbiBUT1BfRklWRV9QUkVTRVRTOg0KICAgICAgICByYWlzZSBLZXlFcnJvcihmInVua25vd24gdG9wLWZpdmUgYXJjaGl0ZWN0dXJlIHtuYW1lIXJ9IikNCiAgICByZXR1cm4gRXZlbnRTcGlrZVNlZ0Jhc2UoDQogICAgICAgIGNsYXNzX2NvdW50PWNsYXNzX2NvdW50LA0KICAgICAgICBpbnB1dF9jaGFubmVscz1pbnB1dF9jaGFubmVscywNCiAgICAgICAgYXJjaGl0ZWN0dXJlPVRPUF9GSVZFX1BSRVNFVFNbbmFtZV0sDQogICAgKQ0K',
    'top_three_models.py': '...',
    'train.py': '...',
}


def _extract_project() -> None:
    package = BUNDLE_DIR / "eventspikeseg"
    package.mkdir(parents=True, exist_ok=True)
    (package / "__init__.py").write_text("", encoding="utf-8")
    for filename, payload in _BUNDLED_MODULES.items():
        (package / filename).write_bytes(base64.b64decode(payload))
    if str(BUNDLE_DIR) not in sys.path:
        sys.path.insert(0, str(BUNDLE_DIR))


def _looks_like_ddd17(path: Path) -> bool:
    layouts = [
        (path / "events" / "train", path / "labels" / "train"),
        (path / "train" / "events", path / "train" / "labels"),
    ]
    for events_dir, labels_dir in layouts:
        if events_dir.is_dir() and labels_dir.is_dir():
            if any(events_dir.glob("*.npy")) and any(labels_dir.glob("*.png")):
                return True
    if (path / "ddd17_train.npz").is_file():
        return True
    return False



HF_REPO = "Chrisathy/BRENet-dataset"
H5_ROOT_OVERRIDE = os.environ.get("DDD17_H5_ROOT", "")
PREPARED_ROOT = Path("/tmp/ddd17_evseg_prepared")
SENSOR_H = 260
SENSOR_W = 346

# BRENet's DDD17 mask palette.
DDD17_PALETTE = {
    (128, 64, 128): 0,   # road / flat
    (70, 70, 70): 1,     # background = construction + sky
    (220, 220, 0): 2,    # object
    (107, 142, 35): 3,   # nature / vegetation
    (220, 20, 60): 4,    # human
    (0, 0, 142): 5,      # vehicle
}


def _ensure_hf_hub() -> None:
    try:
        import huggingface_hub  # noqa: F401
        return
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.27"]
        )
        importlib.invalidate_caches()


def _download_hf_ddd17() -> Path:
    """Use the DDD17 files already downloaded by Kaggle Cell 1."""
    root = HF_RAW_ROOT / "DDD17"
    if not root.is_dir():
        raise FileNotFoundError(
            f"Downloaded DDD17 not found at {root}. "
            "Run the Hugging Face download cell before starting training."
        )
    print(f"Using downloaded Hugging Face DDD17 at {root}")
    return root


def _load_event_index(path: Path) -> np.ndarray:
    """Load BRENet DDD17 frame lookup table: [timestamp_ns, idx_a, idx_b]."""
    raw = np.load(path, allow_pickle=False, mmap_mode="r")
    if raw.ndim != 2 or raw.shape[1] < 1:
        raise ValueError(f"Unexpected index_10ms.npy shape {raw.shape} in {path}")
    if raw.dtype != np.int64:
        raw = np.asarray(raw, dtype=np.int64)
    ts = np.asarray(raw[:, 0], dtype=np.int64)
    if ts.size < 2 or np.any(np.diff(ts) < 0):
        raise ValueError(f"Frame timestamps are not monotonic in {path}")
    return raw


def _open_timestamps(path: Path):
    """BRENet DDD17 stores events.dat.t as one little-endian int64 timestamp per event."""
    size = path.stat().st_size
    if size % 8 != 0:
        raise ValueError(f"events.dat.t size is not divisible by 8: {path}")
    count = size // 8
    arr = np.memmap(path, dtype="<i8", mode="r", shape=(count,))
    # Ignore any zero padding at the physical end of the file.
    if count and arr[-1] == 0:
        nz = np.flatnonzero(np.asarray(arr[-min(count, 1_000_000):]) != 0)
        if nz.size:
            count = count - min(count, 1_000_000) + int(nz[-1]) + 1
            arr = np.memmap(path, dtype="<i8", mode="r", shape=(size // 8,))[:count]
    return arr


def _score_xyz(x, y, p) -> float:
    if len(x) == 0:
        return -1.0
    ok_xy = (
        (x >= 0) & (x < SENSOR_W) &
        (y >= 0) & (y < SENSOR_H)
    )
    ok_p = (p == 0) | (p == 1) | (p == -1)
    return float(np.mean(ok_xy & ok_p))


def _open_xyp(path: Path, event_count: int):
    """
    Open BRENet/ESS events.dat.xyp.

    The routine is deliberately self-validating: it tries the common unpacked
    triplet encodings first, then common packed uint32 bit layouts, and accepts
    only a representation whose sampled x/y/p values are valid DAVIS346 events.
    """
    size = path.stat().st_size
    sample_n = min(event_count, 200_000)

    # Unpacked triplets: [x, y, p].
    #
    # BRENet/ESS files can contain a few trailing padded records. The timestamp
    # loader removes trailing zero timestamps, so xyp may physically contain
    # slightly more records than the logical timestamp/event count. Accept that
    # case and expose only the aligned prefix.
    for dtype in (np.dtype("<i2"), np.dtype("<u2"), np.dtype("<i4"), np.dtype("<u4")):
        record_bytes = 3 * dtype.itemsize
        if size % record_bytes:
            continue
        physical_count = size // record_bytes
        if physical_count < event_count:
            continue

        mm_full = np.memmap(path, dtype=dtype, mode="r", shape=(physical_count, 3))
        mm = mm_full[:event_count]
        s = np.asarray(mm[:sample_n], dtype=np.int64)
        score = _score_xyz(s[:, 0], s[:, 1], s[:, 2])
        if score > 0.995:
            padding = physical_count - event_count
            if padding:
                print(
                    f"  note: ignoring {padding} trailing padded xyp records "
                    f"in {path.name}"
                )
            return ("triplet", mm)

    # Four-byte structured encoding: uint16 x, uint8 y, int/uint8 polarity.
    if size == event_count * 4:
        for p_dtype in ("i1", "u1"):
            dtype = np.dtype([("x", "<u2"), ("y", "u1"), ("p", p_dtype)])
            mm = np.memmap(path, dtype=dtype, mode="r", shape=(event_count,))
            s = mm[:sample_n]
            x = np.asarray(s["x"], dtype=np.int64)
            y = np.asarray(s["y"], dtype=np.int64)
            p = np.asarray(s["p"], dtype=np.int64)
            if _score_xyz(x, y, p) > 0.995:
                return ("structured", mm)

        # Packed uint32. Try typical allocations for DAVIS346 coordinates.
        packed = np.memmap(path, dtype="<u4", mode="r", shape=(event_count,))
        vals = np.asarray(packed[:sample_n], dtype=np.uint32)
        layouts = [
            # x low 9 bits, y next 9 bits, polarity next bit
            (lambda v: (v & 0x1FF, (v >> 9) & 0x1FF, (v >> 18) & 1)),
            # polarity low bit, y next 9, x next 9
            (lambda v: ((v >> 10) & 0x1FF, (v >> 1) & 0x1FF, v & 1)),
            # x high-ish 9, y middle 9, p low
            (lambda v: ((v >> 9) & 0x1FF, (v >> 1) & 0xFF, v & 1)),
        ]
        best = None
        for decoder in layouts:
            x, y, p = decoder(vals)
            score = _score_xyz(
                np.asarray(x, dtype=np.int64),
                np.asarray(y, dtype=np.int64),
                np.asarray(p, dtype=np.int64),
            )
            if best is None or score > best[0]:
                best = (score, decoder)
        if best and best[0] > 0.995:
            return ("packed", (packed, best[1]))

    raise ValueError(
        f"Unsupported events.dat.xyp encoding: {path} "
        f"(size={size}, event_count={event_count})."
    )


def _slice_xyp(handle, start: int, end: int):
    kind, obj = handle
    if kind == "triplet":
        a = np.asarray(obj[start:end], dtype=np.int64)
        return a[:, 0], a[:, 1], a[:, 2]
    if kind == "structured":
        a = obj[start:end]
        return (
            np.asarray(a["x"], dtype=np.int64),
            np.asarray(a["y"], dtype=np.int64),
            np.asarray(a["p"], dtype=np.int64),
        )
    if kind == "packed":
        packed, decoder = obj
        v = np.asarray(packed[start:end], dtype=np.uint32)
        x, y, p = decoder(v)
        return (
            np.asarray(x, dtype=np.int64),
            np.asarray(y, dtype=np.int64),
            np.asarray(p, dtype=np.int64),
        )
    raise RuntimeError(kind)


def _six_channel_evseg(x, y, p, t) -> np.ndarray:
    """EV-SegNet: Hist-, Hist+, Mean-, Mean+, Std-, Std+."""
    out = np.zeros((6, SENSOR_H, SENSOR_W), dtype=np.float32)
    if len(t) == 0:
        return out.astype(np.float16)

    valid = (
        (x >= 0) & (x < SENSOR_W) &
        (y >= 0) & (y < SENSOR_H) &
        ((p == 0) | (p == 1) | (p == -1))
    )
    x, y, p, t = x[valid], y[valid], p[valid], np.asarray(t[valid], dtype=np.float64)
    if len(t) == 0:
        return out.astype(np.float16)

    # Normalize timestamps within the 50 ms integration window.
    t0 = float(t[0])
    t1 = float(t[-1])
    denom = max(t1 - t0, 1.0)
    tn = np.clip((t - t0) / denom, 0.0, 1.0).astype(np.float64)

    # Convert polarity to canonical negative/positive.
    pos_mask = p > 0
    neg_mask = ~pos_mask

    for pair_idx, mask in enumerate((neg_mask, pos_mask)):
        if not np.any(mask):
            continue
        flat = y[mask] * SENSOR_W + x[mask]
        ts = tn[mask]

        count = np.bincount(flat, minlength=SENSOR_H * SENSOR_W).astype(np.float64)
        tsum = np.bincount(flat, weights=ts, minlength=SENSOR_H * SENSOR_W)
        tsq = np.bincount(flat, weights=ts * ts, minlength=SENSOR_H * SENSOR_W)

        mean = np.zeros_like(count)
        nz = count > 0
        mean[nz] = tsum[nz] / count[nz]

        std = np.zeros_like(count)
        gt1 = count > 1
        # Sample standard deviation, matching the EV-SegNet equation (N-1).
        variance = np.zeros_like(count)
        variance[gt1] = (
            tsq[gt1] - (tsum[gt1] * tsum[gt1] / count[gt1])
        ) / (count[gt1] - 1.0)
        std[gt1] = np.sqrt(np.maximum(variance[gt1], 0.0))

        hist = count
        hmax = float(hist.max())
        if hmax > 0:
            hist = hist / hmax

        out[pair_idx] = hist.reshape(SENSOR_H, SENSOR_W)
        out[2 + pair_idx] = mean.reshape(SENSOR_H, SENSOR_W)
        out[4 + pair_idx] = std.reshape(SENSOR_H, SENSOR_W)

    # Float16 roughly halves temporary preprocessing storage; the project loader
    # immediately casts inputs to float32 during training.
    return out.astype(np.float16)


def _convert_mask(src: Path, dst: Path) -> None:
    from PIL import Image
    arr = np.asarray(Image.open(src))
    if arr.ndim == 2:
        out = arr.astype(np.uint8, copy=True)
        invalid = (out > 5) & (out != 255)
        out[invalid] = 255
    elif arr.ndim == 3 and arr.shape[2] >= 3:
        rgb = arr[..., :3].astype(np.uint8)
        out = np.full(rgb.shape[:2], 255, dtype=np.uint8)
        for color, label in DDD17_PALETTE.items():
            hit = np.all(rgb == np.asarray(color, dtype=np.uint8), axis=-1)
            out[hit] = label
    else:
        raise ValueError(f"Unsupported mask shape {arr.shape}: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(out, mode="L").save(dst)


def _index_bins_per_frame(index: np.ndarray, mask_paths: list[Path]) -> int:
    ids = [int(p.stem) for p in mask_paths if p.stem.isdigit()]
    if not ids:
        raise ValueError("DDD17 mask filenames are not numeric")
    max_frame = max(ids)
    if max_frame <= 0:
        return 5

    # index_10ms means the intended value is normally 5 bins per 50 ms frame.
    # Use the sequence length as a sanity check, but do not silently accept a
    # wildly different alignment.
    ratio = (len(index) - 1) / float(max_frame + 1)
    inferred = int(round(ratio))
    if 4 <= inferred <= 6:
        return 5
    # Some index files may include trailing sequence data beyond selected masks.
    # The dataset documentation still specifies 10 ms indexing and EV-SegNet
    # uses a 50 ms integration interval.
    return 5


def _preprocess_sequence(seq_dir: Path, split: str, output_root: Path) -> int:
    masks = sorted((seq_dir / "segmentation_masks").glob("*.png"))
    if not masks:
        return 0

    index_path = seq_dir / "index" / "index_10ms.npy"
    t_path = seq_dir / "events.dat.t"
    xyp_path = seq_dir / "events.dat.xyp"
    for pth in (index_path, t_path, xyp_path):
        if not pth.is_file():
            raise FileNotFoundError(f"Required BRENet DDD17 file missing: {pth}")

    frame_index = _load_event_index(index_path)
    timestamps = _open_timestamps(t_path)
    event_count = len(timestamps)
    xyp = _open_xyp(xyp_path, event_count)

    # Confirm the binary xyp file is exactly one int16 triplet per timestamp.
    expected_xyp_bytes = event_count * 3 * 2
    if xyp_path.stat().st_size < expected_xyp_bytes:
        raise ValueError(
            f"events.dat.xyp is shorter than timestamps for {seq_dir}: "
            f"{xyp_path.stat().st_size} < {expected_xyp_bytes}"
        )
    events_out = output_root / "events" / split
    labels_out = output_root / "labels" / split
    events_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)

    made = 0
    seq_name = seq_dir.name
    print(
        f"Preprocessing {split}/{seq_name}: {len(masks)} masks, "
        f"{event_count:,} events"
    )

    for n, mask_path in enumerate(masks, start=1):
        frame_id = int(mask_path.stem)
        out_stem = f"{seq_name}_{mask_path.stem}"
        event_dst = events_out / f"{out_stem}.npy"
        label_dst = labels_out / f"{out_stem}.png"

        if event_dst.is_file() and label_dst.is_file():
            made += 1
            continue

        # Mask filenames are frame ids into index_10ms.npy. Column 0 is the
        # corresponding frame timestamp in nanoseconds. Build the Ev-SegNet input
        # from the 50 ms of events immediately preceding that semantic label.
        if frame_id < 0 or frame_id >= len(frame_index):
            raise IndexError(
                f"Mask frame id {frame_id} is outside index table of length "
                f"{len(frame_index)} for {seq_dir}"
            )
        target_ts = int(frame_index[frame_id, 0])
        start_ts = target_ts - 50_000_000  # 50 ms in nanoseconds

        start = int(np.searchsorted(timestamps, start_ts, side="left"))
        end = int(np.searchsorted(timestamps, target_ts, side="right"))

        x, y, p = _slice_xyp(xyp, start, end)
        t = np.asarray(timestamps[start:end])
        sample = _six_channel_evseg(x, y, p, t)
        np.save(event_dst, sample, allow_pickle=False)
        _convert_mask(mask_path, label_dst)
        made += 1

        if n % 500 == 0 or n == len(masks):
            print(f"  {n}/{len(masks)}")

    return made


def _prepare_hf_ddd17() -> Path:
    if (
        (PREPARED_ROOT / "events" / "train").is_dir()
        and any((PREPARED_ROOT / "events" / "train").glob("*.npy"))
        and (PREPARED_ROOT / "labels" / "train").is_dir()
    ):
        print(f"Using existing prepared DDD17 at {PREPARED_ROOT}")
        return PREPARED_ROOT

    raw_root = _download_hf_ddd17()
    PREPARED_ROOT.mkdir(parents=True, exist_ok=True)

    totals = {}

    # Canonical training only needs the official DDD17 training split.
    # Validation is derived deterministically from training; the official test
    # split is intentionally reserved for later final evaluation.
    split = "train"
    split_root = raw_root / split
    if not split_root.is_dir():
        raise FileNotFoundError(f"Missing Hugging Face DDD17 split: {split_root}")

    total = 0
    for seq_dir in sorted(p for p in split_root.iterdir() if p.is_dir()):
        total += _preprocess_sequence(seq_dir, split, PREPARED_ROOT)
    totals[split] = total

    print("Prepared DDD17:", totals)
    if totals.get("train", 0) == 0:
        raise RuntimeError("No DDD17 training samples were generated")

    # The project loader may expect test directories to exist during inspection.
    # Create empty placeholders without consuming test data or disk.
    (PREPARED_ROOT / "events" / "test").mkdir(parents=True, exist_ok=True)
    (PREPARED_ROOT / "labels" / "test").mkdir(parents=True, exist_ok=True)

    return PREPARED_ROOT



@dataclass(frozen=True)
class _H5DDD17Sample:
    h5_path: Path
    mask_index: int


def _find_h5_root() -> Path:
    if H5_ROOT_OVERRIDE:
        root = Path(H5_ROOT_OVERRIDE)
        if not root.is_dir():
            raise FileNotFoundError(f"DDD17_H5_ROOT does not exist: {root}")
        return root

    candidates = []
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.is_dir():
            continue
        if (base / "train_dir0.h5").is_file():
            candidates.append(base)
        for p in base.iterdir():
            if p.is_dir() and (p / "train_dir0.h5").is_file():
                candidates.append(p)

    if not candidates:
        # Slightly deeper fallback for Kaggle input wrappers.
        for p in Path("/kaggle/input").rglob("train_dir0.h5"):
            candidates.append(p.parent)
            break

    if not candidates:
        raise FileNotFoundError(
            "Could not find DDD17 HDF5 files under /kaggle/input or /kaggle/working. "
            "Attach the ddd17-brenet-h5 Kaggle Dataset first."
        )
    return candidates[0]


class _H5StreamingDDD17Dataset:
    """Build six-channel Ev-SegNet samples on demand from sequence HDF5 files."""

    def __init__(self, samples, *, output_size):
        import torch
        self.samples = list(samples)
        self.output_size = output_size
        self._files = {}
        self._torch = torch

    def __len__(self):
        return len(self.samples)

    def _file(self, path: Path):
        import h5py
        f = self._files.get(path)
        if f is None:
            f = h5py.File(path, "r")
            self._files[path] = f
        return f

    def _mask_tensor(self, sample):
        import torch
        from torch.nn import functional as F

        f = self._file(sample.h5_path)
        mask = np.asarray(f["masks"][sample.mask_index], dtype=np.int64)
        tensor = torch.from_numpy(mask).long()

        if self.output_size is not None:
            tensor = F.interpolate(
                tensor[None, None].float(),
                size=self.output_size,
                mode="nearest",
            )[0, 0].long()

        return tensor

    def mask_only(self, index: int):
        return self._mask_tensor(self.samples[index])

    def __getitem__(self, index):
        import torch
        from torch.nn import functional as F

        sample = self.samples[index]
        f = self._file(sample.h5_path)

        start, end = np.asarray(
            f["event_ranges"][sample.mask_index], dtype=np.int64
        ).tolist()

        t = np.asarray(f["timestamps"][start:end], dtype=np.int64)
        xyp = np.asarray(f["xyp"][start:end], dtype=np.int16)

        if xyp.ndim != 2 or xyp.shape[1] != 3:
            raise ValueError(
                f"Bad xyp slice shape {xyp.shape} in {sample.h5_path}"
            )

        event_np = _six_channel_evseg(
            xyp[:, 0].astype(np.int64, copy=False),
            xyp[:, 1].astype(np.int64, copy=False),
            xyp[:, 2].astype(np.int64, copy=False),
            t,
        ).astype(np.float32, copy=False)

        events = torch.from_numpy(event_np).unsqueeze(0)  # [1,6,H,W]

        if self.output_size is not None:
            _, c, _, _ = events.shape
            events = F.interpolate(
                events.view(c, 1, *events.shape[-2:]),
                size=self.output_size,
                mode="area",
            ).view(1, c, *self.output_size)

        mask = self._mask_tensor(sample)
        return events, mask


def _h5_train_samples(h5_root: Path):
    import h5py

    files = sorted(h5_root.glob("train_*.h5"))
    if not files:
        raise FileNotFoundError(f"No train_*.h5 files found under {h5_root}")

    samples = []

    for path in files:
        with h5py.File(path, "r") as f:
            if not bool(f.attrs.get("complete", False)):
                raise RuntimeError(f"Incomplete HDF5 sequence: {path}")
            count = int(f["masks"].shape[0])

        samples.extend(
            _H5DDD17Sample(path, i) for i in range(count)
        )

    if not samples:
        raise RuntimeError(f"No HDF5 DDD17 training samples found under {h5_root}")

    return samples


def _streaming_prepare_datasets(
    root,
    *,
    seed,
    train_samples,
    output_size,
    final_validation_fraction=None,
):
    import torch

    h5_root = _find_h5_root()
    samples = _h5_train_samples(h5_root)

    if train_samples <= 0:
        selected_count = len(samples)
    elif train_samples <= 1:
        selected_count = max(1, int(round(len(samples) * train_samples)))
    else:
        if not float(train_samples).is_integer():
            raise ValueError("train_samples > 1 must be an integer-valued count")
        selected_count = min(len(samples), int(train_samples))

    if selected_count < len(samples):
        g = torch.Generator().manual_seed(seed)
        indices = torch.randperm(len(samples), generator=g)[:selected_count].tolist()
        selected = [samples[i] for i in indices]
    else:
        selected = list(samples)

    if final_validation_fraction is None:
        raise ValueError(
            "HDF5 Kaggle launcher is for the canonical final suite only; "
            "a validation fraction is required."
        )
    if not (0.0 < final_validation_fraction < 0.5):
        raise ValueError("validation_fraction must satisfy 0 < value < 0.5")
    if len(selected) < 2:
        raise ValueError("At least two selected DDD17 samples are required.")

    g = torch.Generator().manual_seed(seed + 1)
    order = torch.randperm(len(selected), generator=g).tolist()

    validation_count = max(
        1, int(round(len(selected) * final_validation_fraction))
    )
    validation_count = min(validation_count, len(selected) - 1)

    val_samples = [selected[i] for i in order[:validation_count]]
    train_samples_list = [selected[i] for i in order[validation_count:]]

    print(
        f"DDD17 HDF5 streaming dataset: root={h5_root}, total={len(samples)}, "
        f"selected={len(selected)}, train={len(train_samples_list)}, "
        f"validation={len(val_samples)}"
    )

    return (
        _H5StreamingDDD17Dataset(train_samples_list, output_size=output_size),
        _H5StreamingDDD17Dataset(val_samples, output_size=output_size),
    )


def _streaming_estimate_class_weights(dataset, maximum=None):
    """Estimate class weights from HDF5 masks only; event arrays are not read."""
    import torch

    counts = torch.zeros(6, dtype=torch.float64)
    limit = len(dataset) if maximum is None else min(len(dataset), maximum)

    print(f"Estimating class weights from {limit} HDF5 masks...")
    for index in range(limit):
        mask = dataset.mask_only(index)
        valid = mask != 255
        if valid.any():
            counts += torch.bincount(mask[valid], minlength=6)

    frequencies = counts / counts.sum().clamp_min(1.0)
    weights = 1.0 / torch.sqrt(frequencies.clamp_min(1e-6))
    weights /= weights.mean().clamp_min(1e-8)
    return weights.float().clamp(max=8.0)

def _find_ddd17_root() -> Path:
    if DDD17_ROOT:
        root = Path(DDD17_ROOT)
        if not root.exists():
            raise FileNotFoundError(f"DDD17_ROOT does not exist: {root}")
        return root

    search_roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    candidates = []
    first_level = []

    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        candidates.append(search_root)
        try:
            children = [p for p in search_root.iterdir() if p.is_dir()]
        except OSError:
            children = []
        first_level.extend(children)
        candidates.extend(children)
        for p in children:
            try:
                candidates.extend(q for q in p.iterdir() if q.is_dir())
            except OSError:
                pass

    # Prefer an exact DDD17-looking root.
    for candidate in candidates:
        if _looks_like_ddd17(candidate):
            return candidate

    # Let the project loader inspect common wrapper roots.
    for candidate in first_level:
        for name in (
            "DDD17", "ddd17", "dataset", "our_codification",
            "dataset_DDD17-Events_our_codification",
        ):
            nested = candidate / name
            if nested.is_dir():
                return nested

    raise FileNotFoundError(
        "Could not find prepared DDD17 data under /kaggle/input or /kaggle/working. "
        "Extract the downloaded DDD17 ZIP first."
    )


def _ensure_spikingjelly() -> None:
    try:
        import spikingjelly  # noqa: F401
        return
    except ImportError:
        pass

    print("SpikingJelly is not installed; attempting the project-pinned install...")
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "spikingjelly==0.0.0.0.14",
    ]
    try:
        subprocess.check_call(cmd)
        importlib.invalidate_caches()
        import spikingjelly  # noqa: F401
    except Exception as exc:
        raise RuntimeError(
            "SpikingJelly installation failed. In Kaggle, enable Internet for the notebook "
            "and rerun this script, or add a SpikingJelly wheel as a Kaggle Input. "
            "The canonical project pins spikingjelly==0.0.0.0.14."
        ) from exc



def _enable_safe_gradient_checkpointing() -> None:
    """
    Reduce activation VRAM without checkpointing any stateful spiking-neuron logic.

    EventSpikeSeg's SpikingStage has two conceptually separate parts:
      1. stateless Conv2d + affine/identity current transform over flattened T*N
      2. sequential HardwareLIF/PLIF neuron updates over T

    Only (1) is recomputed during backward. This deliberately avoids wrapping
    HardwareLIF/PLIF, f_inst, f_long, adaptive thresholds, membrane state, or
    reset logic in torch.utils.checkpoint.
    """
    import torch
    from torch.utils.checkpoint import checkpoint
    import eventspikeseg.model_components as mc

    if getattr(mc.SpikingStage, "_safe_checkpoint_patch", False):
        return

    original_forward = mc.SpikingStage.forward

    def checkpointed_forward(self, sequence):
        if sequence.ndim != 5:
            raise ValueError("expected a TNCHW sequence")

        # Preserve historical BatchNorm semantics exactly. The canonical final
        # architecture uses affine normalization, so this branch is normally
        # irrelevant, but BN has running-state updates and must not be replayed.
        if self.norm_kind == "bn":
            return original_forward(self, sequence)

        time_steps, batch_size = sequence.shape[:2]
        flat = sequence.reshape(
            time_steps * batch_size, *sequence.shape[2:]
        ).float()

        def stateless_current_transform(x):
            return self.transform(self.conv(x))

        # During training, recompute only Conv/current-transform activations in
        # backward instead of storing their internal graph. use_reentrant=False
        # is the modern PyTorch checkpoint implementation and correctly tracks
        # parameters captured by this closure.
        if self.training and torch.is_grad_enabled():
            currents = checkpoint(
                stateless_current_transform,
                flat,
                use_reentrant=False,
                preserve_rng_state=False,
            )
        else:
            currents = stateless_current_transform(flat)

        currents = currents.reshape(
            time_steps, batch_size, *currents.shape[1:]
        )

        # CRITICAL: neuron dynamics remain outside checkpointing and execute
        # exactly once in forward, preserving membrane/adaptation state.
        return torch.stack([
            self.neuron(current) for current in currents
        ])

    mc.SpikingStage.forward = checkpointed_forward
    mc.SpikingStage._safe_checkpoint_patch = True
    print(
        "Gradient checkpointing: ENABLED "
        "(stateless Conv/current transforms only; neuron state is not replayed)"
    )

def main() -> None:
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
    _extract_project()
    _ensure_spikingjelly()
    _enable_safe_gradient_checkpointing()

    import torch
    import eventspikeseg.train as train_module

    h5_root = _find_h5_root()
    raw_root = h5_root

    # Critical Kaggle storage fix: patch only the dataset construction and
    # class-weight scan. The project's model, losses, optimizer, scheduler,
    # checkpointing, validation, and final-suite training loop remain unchanged.
    train_module._prepare_datasets = _streaming_prepare_datasets
    train_module.estimate_class_weights = _streaming_estimate_class_weights

    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = bool(USE_AMP and device == "cuda")

    print("=" * 72)
    print("EventSpikeSeg canonical DDD17 STREAMING training")
    print(f"HDF5 dataset  : {h5_root}")
    print(f"Device        : {device}")
    if device == "cuda":
        print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"AMP           : {amp}")
    print(f"Architecture  : {ARCHITECTURE}")
    print(f"Epochs        : {EPOCHS}")
    print(f"Input size    : {HEIGHT}x{WIDTH}")
    print(f"Time steps    : {TIME_STEPS}")
    print(f"Train subset  : {TRAIN_SAMPLES:.0%}")
    print(f"Val holdout   : {VALIDATION_FRACTION:.0%}")
    print("Preproc cache : DISABLED (samples built on demand)")
    print(f"Output        : {OUTPUT_DIR}")
    print("=" * 72)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Resume automatically from the newest completed epoch checkpoint.
    periodic = list(OUTPUT_DIR.glob(f"{ARCHITECTURE}_seed{SEED}_epoch*.pt"))
    resume_path = None

    if periodic:
        def _epoch_number(path):
            try:
                return int(path.stem.rsplit("epoch", 1)[1])
            except Exception:
                return -1

        periodic.sort(key=_epoch_number)
        candidate = periodic[-1]

        if _epoch_number(candidate) > 0:
            resume_path = candidate
            print(f"Auto-resume    : {resume_path}")
        else:
            print("Auto-resume    : no valid epoch checkpoint found")
    else:
        print("Auto-resume    : no checkpoint found; starting from epoch 1")

    result = train_module.train_architecture(
        raw_root,
        architecture=ARCHITECTURE,
        suite=SUITE,
        seed=SEED,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        time_steps=TIME_STEPS,
        train_samples=TRAIN_SAMPLES,
        validation_fraction=VALIDATION_FRACTION,
        learning_rate=LEARNING_RATE,
        output_dir=OUTPUT_DIR,
        output_size=(HEIGHT, WIDTH),
        device=device,
        checkpoint_every=CHECKPOINT_EVERY,
        amp=amp,
        load=resume_path,
    )

    print("\nTraining complete.")
    print(result.to_dict())
    print("\nCheckpoints:")
    print(OUTPUT_DIR)


if __name__ == "__main__":
    main()


Training PyTorch: 2.10.0+cu126
Training CUDA: 12.6
GPU: Tesla P100-PCIE-16GB
Architectures: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']
EventSpikeSeg canonical DDD17 STREAMING training
HDF5 dataset  : /kaggle/input/datasets/stamir/ddd17-brenet-h5
Device        : cuda
GPU           : Tesla P100-PCIE-16GB
AMP           : True
Architecture  : EventSpikeSeg
Epochs        : 70
Input size    : 212x282
Time steps    : 8
Train subset  : 80%
Val holdout   : 10%
Preproc cache : DISABLED (samples built on demand)
Output        : /kaggle/working/eventspikeseg_checkpoints
Auto-resume    : no checkpoint found; starting from epoch 1
DDD17 HDF5 streaming dataset: root=/kaggle/input/datasets/stamir/ddd17-brenet-h5, total=15950, selected=12760, train=11484, validation=1276
Estimating class weights from 11484 HDF5 masks...


/kaggle/working/_eventspikeseg_bundle/eventspikeseg/train.py:470: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


Training EventSpikeSeg | cuda | AMP FP16 | epochs=70 batch=1 T=8 size=212x282 | train=11484 val=1276 | params=237,597


Epoch 01/70:   0%|          | 0/11484 [00:00<?, ?batch/s]

/kaggle/working/_eventspikeseg_bundle/eventspikeseg/train.py:740: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled, dtype=torch.float16):
/kaggle/working/torch-cu126/torch/nn/functional.py:3504: UserWarning: nll_loss2d_forward_out_cuda_template does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return torch._C._nn.cross_entropy_loss(
